## Requirements

In [ ]:
import numpy as np
from Min3D.core.simulators import (
    add_gaussian_noise,
    generate_hollow_cylinder,
)
from Min3D.core.util import distance_from_line
from Min3D import GeometryTransformationTool, SurfaceGraph, shortest_path_interactive

## Generate Point Cloud - Hollow Cylinder

In [ ]:
## Generate Point Cloud - Hollow Cylinder & Add Noise
h_cylinder = generate_hollow_cylinder(
    center=(0, 0, 0), inner_radius=1.9, outer_radius=2, height=40, num_points=20000
)
h_cylinder = add_gaussian_noise(h_cylinder, sigma=0.25)
h_cylinder.paint(distance_from_line(h_cylinder), cmap="Reds")

In [ ]:
# Visualize
h_cylinder.visualize()

## Surface Reconstruction

In [ ]:
tool = GeometryTransformationTool()

# Convex Hull
convex_mesh = tool.convex_hull_from(h_cylinder)

# Concave Hull
concave_mesh = (
    (tool.concave_hull_from(h_cylinder, alpha=0.8))
    .make_watertight()
    .clean_and_cluster()
)

# Have a look at the results
print(f"Convex Hull:\n{str(convex_mesh)}")
print("\n" + "=" * 50 + "\n")
print(f"Concave Hull:\n{str(concave_mesh)}")

### Let's take a look at the meshes
Try to spot the differences between the convex and concave meshes. The convex mesh should be a tight-fitting hull around the point cloud, while the concave mesh should capture the hollow structure of the cylinder.

In [ ]:
# Let's take a look at the meshes
convex_mesh.visualize()

In [ ]:
# Let's take a look at the meshes
concave_mesh.visualize()

## Sample the Surface to get a new tightly fitting point cloud

In [ ]:
## Sample the Surface to get a new tightly fitting point cloud

# uniform sampling
un_pcd = tool.sample_uniformly_from(concave_mesh, number_of_points=10000)

# poisson disk sampling
pd_pcd = tool.sample_poisson_disk_from(concave_mesh, number_of_points=10000)

# Let's visualize the distance to the nearest neighbor for both point clouds to see how well they capture the surface
pd_pcd.plot_dNN_for(un_pcd, pd_pcd, x_lim=(0, 0.4))

# Is our resolution ok? Let's take a look at the surface area.
print(f"Surface Area of Concave Mesh: {concave_mesh.get_surface_area():.2f}")

# What's the main difference between the two sampling methods?
# The uniform sampling should give you a more even distribution of points across the surface,
# while the poisson disk sampling should give you a more natural, clustered distribution that captures the geometry of the surface better.

# As we want to map the surface and accurately capture the geometry, the poisson disk sampling is likely the better choice here.
# The uniform sampling may miss some of the finer details of the surface, while the poisson disk sampling should capture those details more effectively.

# Plus, with the tight grouping of the points in the poisson disk sampling, we can expect a homogeneous surface resolution, which is important for downstream applications like meshing and pathfinding.

## Make Surface Wireframe - from kNN

In [ ]:
# Make Surface Wireframe - from kNN
surface_wireframe = tool.kNN_wireframe_from(
    pd_pcd, k=6
)  # <- we choose k=6 to ensure a good balance between connectivity and sparsity in the wireframe. Too low of a k might result in a disconnected graph, while too high of a k could lead to an overly dense graph that doesn't capture the surface structure well.

# Visualize the wireframe
surface_wireframe.visualize_with(pd_pcd)

## Make Graph from Wireframe

In [ ]:
# Now we can convert the wireframe into a surface graph, which will allow us to perform various graph-based analyses and operations on the surface structure.
surface_graph = SurfaceGraph.from_wireframe(surface_wireframe)

In [ ]:
# It's time for pathfinding - let's take a look at a path on the surface
start_node: int = 2
end_node: int = 200

# check if the nodes exist in the graph
if not start_node >= 0 and start_node < surface_graph.graph.num_nodes():
    raise ValueError(
        f"Start node {start_node} is out of bounds for the graph with {surface_graph.graph.num_nodes} nodes."
    )

if not end_node >= 0 and end_node < surface_graph.graph.num_nodes():
    raise ValueError(
        f"End node {end_node} is out of bounds for the graph with {surface_graph.graph.num_nodes} nodes."
    )

# Interactive shortest path visualization
shortest_path_interactive(surface_graph, source=start_node, target=end_node)

# Let's check if the euclidean distance between the points is a good approximation of the geodesic distance along the surface.
# We can do this by comparing the length of the shortest path on the graph to the euclidean distance between the start and end points in the point cloud.
geodesic_distance = surface_graph.get_shortest_distance(
    source=start_node, target=end_node
)
euclidean_distance = np.linalg.norm(
    surface_graph.vertices.points[start_node] - surface_graph.vertices.points[end_node]
)

print("Geodesic Distance vs Euclidean Distance:")
print(f"Start Node: {start_node}, End Node: {end_node}")
print(f"Geodesic Distance: {geodesic_distance:.2f}")
print(f"Euclidean Distance: {euclidean_distance:.2f}")
print(f"Ratio (Geodesic / Euclidean): {geodesic_distance / euclidean_distance:.2f}")